# Bank Good Credit Prediction
### PM-PR-1005-Client Project

**Domain:** Banking-Risk

**Objective:** To develop a predictive model that helps **Bank GoodCredit** identify customers likely to default on credit payments and reduce credit risk.


# ***1. Bussiness Case***
Bank GoodCredit wants to predict cred score for current credit card
customers. The cred score will denote a customer’s credit worthiness
and help the bank in reducing credit default risk

## 2. Domain Analysis
**Domain:Banking – Credit Risk Analytics**

**Problem Type: Binary Classification**

- Dataset is anonymized

- Multiple behavioural and demographic features (~66+ engineered features)

- customer_no: Unique customer identifier

- Data collected from account history, enquiry behaviour, and customer profile

* Target variable:

- 0 → Customer has Good credit history

- 1 → Customer has Bad credit history (30+ DPD)

Detailed EDA is limited due to anonymized and privacy-protected features provided by Bank GoodCredit.

## Task 1: Data Analysis Report

In [4]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 34.2 MB/s eta 0:00:00


**1. IMPORT LIBRARIES**

In [5]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Database

import mysql.connector

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Metrics
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

**Connect to Database**

In [6]:
conn = mysql.connector.connect(
    host="18.136.157.135",
    user="dm_team1",
    password="DM!$Team&279@20!",
    database="project_banking"
)

cust_account = pd.read_sql("SELECT * FROM Cust_Account", conn)
cust_enquiry = pd.read_sql("SELECT * FROM Cust_Enquiry", conn)
cust_demo = pd.read_sql("SELECT * FROM Cust_Demographics", conn)

conn.close()

/tmp/ipython-input-3560381621.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cust_account = pd.read_sql("SELECT * FROM Cust_Account", conn)
/tmp/ipython-input-3560381621.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cust_enquiry = pd.read_sql("SELECT * FROM Cust_Enquiry", conn)
/tmp/ipython-input-3560381621.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cust_demo = pd.read_sql("SELECT * FROM Cust_Demographics", conn)


**Basic Checks**

In [7]:
cust_account.head()


,dt_opened,customer_no,upload_dt,acct_type,owner_indic,opened_dt,last_paymt_dt,closed_dt,reporting_dt,high_credit_amt,...,amt_past_due,paymenthistory1,paymenthistory2,paymt_str_dt,paymt_end_dt,creditlimit,cashlimit,rateofinterest,paymentfrequency,actualpaymentamount
0,10-Nov-15,12265,20-Oct-15,6,1,09-Jun-13,30-Jun-14,05-Jul-14,30-Sep-15,20900,...,,"""""""STDSTDSTDXXXXXXXXXXXXXXXSTDXXXXXXXXXXXXXXXS...",,01-Sep-15,01-Jul-14,,,,,
1,10-Nov-15,12265,20-Oct-15,10,1,25-May-12,06-Sep-15,,03-Oct-15,16201,...,,"""""""0000000000000000000000000000000000000000000...","""""""000000000000000000000000000XXX0000000000000...",01-Oct-15,01-Nov-12,14000,1400,,3,5603
2,10-Nov-15,12265,20-Oct-15,10,1,22-Mar-12,31-Aug-15,,30-Sep-15,41028,...,,"""""""0000000000000000000000000000000000000000000...","""""""0000000000000000000000000000000000000000000...",01-Sep-15,01-Oct-12,,,,,
3,20-Jul-15,15606,09-Jul-15,10,1,13-Jan-06,,26-Jul-07,31-Jan-09,93473,...,,"""""""1200900600600600300000000000000000000000000...",,01-Jul-07,01-Feb-06,,,,,
4,20-Jul-15,15606,09-Jul-15,6,1,18-Jan-15,05-May-15,,31-May-15,20250,...,,"""""""000000000000000""""""",,01-May-15,01-Jan-15,,,,,


In [8]:
cust_enquiry.head()

,dt_opened,customer_no,upload_dt,enquiry_dt,enq_purpose,enq_amt
0,18-Apr-15,1,21-Apr-15,19-Dec-14,2,3500000
1,18-Apr-15,1,21-Apr-15,05-Mar-14,5,500000
2,18-Apr-15,1,21-Apr-15,05-Mar-14,0,50000
3,18-Apr-15,1,21-Apr-15,22-Feb-14,10,50000
4,18-Apr-15,1,21-Apr-15,11-Jun-13,10,1000


In [9]:
cust_demo.head()

,dt_opened,customer_no,entry_time,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,...,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,Bad_label
0,18-Apr-15,1,13-Apr-15,Insignia,13-Apr-15,650,2,Card Setup,14,500000,...,21,R,,,0000-00-00,0,98332XXXXX,1,N,0
1,21-Apr-15,2,21-Apr-15,Insignia,21-Apr-15,760,1,Card Setup,14,1200000,...,17,R,,,0000-00-00,0,99455XXXXX,1,N,0
2,22-Apr-15,3,21-Apr-15,Insignia,21-Apr-15,774,1,Card Setup,14,700000,...,17,R,,,0000-00-00,0,98456XXXXX,1,N,0
3,25-Apr-15,4,15-Apr-15,Insignia,20-Apr-15,770,1,Card Setup,14,500000,...,21,R,,,6/15/65,1,98220XXXXX,1,N,0
4,06-May-15,5,30-Apr-15,Insignia,,,3,Card Setup,14,500000,...,13,R,,,0000-00-00,0,98111XXXXX,1,N,0


In [10]:
cust_account.shape

(186329, 21)

In [11]:
cust_enquiry.shape

(413188, 6)

In [12]:
cust_demo.shape

(23896, 83)

In [13]:
cust_account.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186329 entries, 0 to 186328
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   dt_opened            186329 non-null  object
 1   customer_no          186329 non-null  object
 2   upload_dt            186329 non-null  object
 3   acct_type            186329 non-null  object
 4   owner_indic          186329 non-null  object
 5   opened_dt            186329 non-null  object
 6   last_paymt_dt        186329 non-null  object
 7   closed_dt            186329 non-null  object
 8   reporting_dt         186329 non-null  object
 9   high_credit_amt      186329 non-null  object
 10  cur_balance_amt      186329 non-null  object
 11  amt_past_due         186329 non-null  object
 12  paymenthistory1      186329 non-null  object
 13  paymenthistory2      186329 non-null  object
 14  paymt_str_dt         186329 non-null  object
 15  paymt_end_dt         186329 non-nu

In [14]:
cust_enquiry.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 413188 entries, 0 to 413187
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   dt_opened    413188 non-null  object
 1   customer_no  413188 non-null  object
 2   upload_dt    413188 non-null  object
 3   enquiry_dt   413188 non-null  object
 4   enq_purpose  413188 non-null  object
 5   enq_amt      413188 non-null  object
dtypes: object(6)
memory usage: 18.9+ MB


In [15]:
cust_demo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23896 entries, 0 to 23895
Data columns (total 83 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   dt_opened    23896 non-null  object
 1   customer_no  23896 non-null  object
 2   entry_time   23896 non-null  object
 3   feature_1    23896 non-null  object
 4   feature_2    23896 non-null  object
 5   feature_3    23896 non-null  object
 6   feature_4    23896 non-null  object
 7   feature_5    23896 non-null  object
 8   feature_6    23896 non-null  object
 9   feature_7    23896 non-null  object
 10  feature_8    23896 non-null  object
 11  feature_9    23896 non-null  object
 12  feature_10   23896 non-null  object
 13  feature_11   23896 non-null  object
 14  feature_12   23896 non-null  object
 15  feature_13   23896 non-null  object
 16  feature_14   23896 non-null  object
 17  feature_15   23896 non-null  object
 18  feature_16   23896 non-null  object
 19  feature_17   23896 non-nu

In [16]:
cust_account.describe()


,dt_opened,customer_no,upload_dt,acct_type,owner_indic,opened_dt,last_paymt_dt,closed_dt,reporting_dt,high_credit_amt,...,amt_past_due,paymenthistory1,paymenthistory2,paymt_str_dt,paymt_end_dt,creditlimit,cashlimit,rateofinterest,paymentfrequency,actualpaymentamount
count,186329,186329,186329,186329,186329,186329,186329,186329,186329,186329,...,186329,186329,186329,186329,186329,186329,186329,186329,186329,186329
unique,197,23896,75,31,4,6246,4511,4840,1877,67555,...,630,18311,11912,234,235,1380,1456,1476,3,18300
top,16-Nov-15,8516,21-Apr-15,10,1,13-Apr-12,,,31-Jul-15,,...,,"""""""0000000000000000000000000000000000000000000...",,01-Jul-15,01-Jan-15,,,,,
freq,4643,120,5128,100239,177287,514,25487,109075,12654,8875,...,185453,44979,107824,16747,5640,137477,151047,161496,122436,145276


In [17]:
cust_enquiry.describe()

,dt_opened,customer_no,upload_dt,enquiry_dt,enq_purpose,enq_amt
count,413188,413188,413188,413188,413188,413188
unique,197,23896,76,3772,37,7384
top,16-Nov-15,10076,21-Apr-15,17-Mar-15,10,50000
freq,10538,308,11770,495,238150,79949


In [18]:
cust_demo.describe()

,dt_opened,customer_no,entry_time,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,...,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,Bad_label
count,23896,23896,23896,23896,23896,23896,23896,23896,23896,23896,...,23896,23896,23896,23896,23896,23896,23896,23896,23896,23896
unique,197,23896,297,8,282,263,4,2,2,485,...,14,3,3,4,63,6,3124,4,3,2
top,16-Nov-15,23896,11-Sep-15,Platinum Maxima,,,3,Card Setup,14,35000,...,10,R,,,0000-00-00,0,XXXXX,1,N,0
freq,699,1,180,9056,2836,2836,14593,23881,23881,797,...,9760,15617,20951,23879,23820,23817,2149,22958,23876,22892


In [19]:
cust_account.isnull().sum()

,0
dt_opened,0
customer_no,0
upload_dt,0
acct_type,0
owner_indic,0
opened_dt,0
last_paymt_dt,0
closed_dt,0
reporting_dt,0
high_credit_amt,0


In [20]:
cust_enquiry.isnull().sum()

,0
dt_opened,0
customer_no,0
upload_dt,0
enquiry_dt,0
enq_purpose,0
enq_amt,0


In [21]:
cust_demo.isnull().sum()

,0
dt_opened,0
customer_no,0
entry_time,0
feature_1,0
feature_2,0
...,...
feature_76,0
feature_77,0
feature_78,0
feature_79,0


In [22]:
cust_account.duplicated().sum()

np.int64(2401)

In [23]:
cust_enquiry.duplicated().sum()

np.int64(13827)

In [24]:
cust_demo.duplicated().sum()

np.int64(0)

In [25]:
cust_account.duplicated().sum()

np.int64(2401)

In [26]:
cust_account = cust_account.drop_duplicates()


In [27]:
cust_account.duplicated().sum()


np.int64(0)

In [28]:
cust_enquiry.duplicated().sum()

np.int64(13827)

In [29]:
cust_enquiry = cust_enquiry.drop_duplicates()

In [30]:
cust_enquiry.duplicated().sum()

np.int64(0)

**DATA CLEANING**

In [31]:
cust_account.columns = cust_account.columns.str.lower()
cust_enquiry.columns = cust_enquiry.columns.str.lower()
cust_demo.columns = cust_demo.columns.str.lower()

In [32]:
# Convert date columns
date_cols_acc = ['opened_dt','last_paymt_dt','closed_dt','reporting_dt']
for col in date_cols_acc:
    cust_account[col] = pd.to_datetime(cust_account[col], errors='coerce')

cust_enquiry['enquiry_dt'] = pd.to_datetime(cust_enquiry['enquiry_dt'], errors='coerce')


/tmp/ipython-input-3916473035.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_account[col] = pd.to_datetime(cust_account[col], errors='coerce')
/tmp/ipython-input-3916473035.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_account[col] = pd.to_datetime(cust_account[col], errors='coerce')
/tmp/ipython-input-3916473035.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cust_account[col] = pd.to_datetime(cust_account[col], errors='coerce')
/tmp/ipython-input-3916473035.py:4: UserWarning: Could not infer format, so each element will be parsed individually, fal

In [ ]:
acc_features

,customer_no,high_credit_amt,cur_balance_amt,amt_past_due,creditlimit,cashlimit,ratio_currbalance_creditlimit
0,1,526670.866667,261936.500000,1269104.5,335000.000000,168000.0,0.162523
1,10,828780.000000,20643.250000,NaN,405000.000000,243000.0,0.441295
2,100,106085.111111,48269.444444,NaN,62333.333333,13750.0,0.360920
3,1000,414755.333333,20154.000000,NaN,247500.000000,74250.0,0.069664
4,10000,189835.666667,181026.333333,NaN,60000.000000,10000.0,0.292112
...,...,...,...,...,...,...,...
23891,9995,100230.800000,77181.600000,NaN,43900.000000,9800.0,0.979305
23892,9996,58822.000000,44114.500000,NaN,65000.000000,13000.0,0.050938
23893,9997,66024.000000,1916.600000,NaN,NaN,NaN,NaN
23894,9998,26505.333333,24171.333333,NaN,28000.000000,4100.0,0.934274


In [35]:
# FIX DATA TYPES

# Account numeric columns
acc_numeric_cols = [
    'high_credit_amt',
    'cur_balance_amt',
    'amt_past_due',
    'creditlimit',
    'cashlimit',
    'rateofinterest',
    'actualpaymentamount'
]

for col in acc_numeric_cols:
    cust_account[col] = pd.to_numeric(cust_account[col], errors='coerce')


# Enquiry numeric columns
cust_enquiry['enq_amt'] = pd.to_numeric(cust_enquiry['enq_amt'], errors='coerce')


In [36]:
acc_numeric_cols

['high_credit_amt',
 'cur_balance_amt',
 'amt_past_due',
 'creditlimit',
 'cashlimit',
 'rateofinterest',
 'actualpaymentamount']

In [38]:
cust_account['payment_duration'] = (
    cust_account['last_paymt_dt'] - cust_account['opened_dt']
).dt.days

pay_features = cust_account.groupby('customer_no')['payment_duration'].mean().reset_index()

In [39]:
 # ENQUIRY FEATURES

cust_enquiry['days_since_enquiry'] = (
    pd.Timestamp.today() - cust_enquiry['enquiry_dt']
).dt.days

enq_features = cust_enquiry.groupby('customer_no').agg({
    'enq_amt':'mean',
    'days_since_enquiry':'mean'
}).reset_index()

In [40]:
# MERGING DATA

df = cust_demo.merge(acc_features, on='customer_no', how='left')
df = df.merge(pay_features, on='customer_no', how='left')
df = df.merge(enq_features, on='customer_no', how='left')

In [41]:
df.to_csv('bnk_credit.csv')

In [42]:
df.fillna(0, inplace=True)

/tmp/ipython-input-4231983114.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df.fillna(0, inplace=True)


In [43]:
df.head()

,dt_opened_x,customer_no,entry_time,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,...,paymt_str_dt,paymt_end_dt,creditlimit,cashlimit,rateofinterest,paymentfrequency,actualpaymentamount,payment_duration,enq_amt,days_since_enquiry
0,18-Apr-15,1,13-Apr-15,Insignia,13-Apr-15,650,2,Card Setup,14,500000,...,01-Feb-15,01-Mar-12,,,,,,1496.0,276730.555556,5400.5
1,18-Apr-15,1,13-Apr-15,Insignia,13-Apr-15,650,2,Card Setup,14,500000,...,01-Feb-15,01-Feb-14,250000,,,3,25100,1496.0,276730.555556,5400.5
2,18-Apr-15,1,13-Apr-15,Insignia,13-Apr-15,650,2,Card Setup,14,500000,...,01-Mar-09,01-Jul-07,,,,,,1496.0,276730.555556,5400.5
3,18-Apr-15,1,13-Apr-15,Insignia,13-Apr-15,650,2,Card Setup,14,500000,...,01-Jan-07,01-Dec-05,,,,,,1496.0,276730.555556,5400.5
4,18-Apr-15,1,13-Apr-15,Insignia,13-Apr-15,650,2,Card Setup,14,500000,...,01-May-10,01-May-08,,,,,,1496.0,276730.555556,5400.5


In [44]:
df = df.drop_duplicates()


In [45]:
df['customer_no'].duplicated().sum()

np.int64(160032)

In [46]:
df = df.drop_duplicates(keep='last')


In [47]:
df.columns

Index(['dt_opened_x', 'customer_no', 'entry_time', 'feature_1', 'feature_2',
       'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7',
       ...
       'paymt_str_dt', 'paymt_end_dt', 'creditlimit', 'cashlimit',
       'rateofinterest', 'paymentfrequency', 'actualpaymentamount',
       'payment_duration', 'enq_amt', 'days_since_enquiry'],
      dtype='object', length=106)

**Missing Values**

In [48]:
df.isnull().sum()

,0
dt_opened_x,0
customer_no,0
entry_time,0
feature_1,0
feature_2,0
...,...
paymentfrequency,0
actualpaymentamount,0
payment_duration,0
enq_amt,0


In [49]:
!pip install scorecardpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for scorecardpy: filename=scorecardpy-0.1.9.7-py3-none-any.whl size=60629 sha256=76b2267376abb608017b12bd59894a0c41f23954e85a54c779b870adb86fec2c
  Stored in directory: /root/.cache/pip/wheels/9f/d8/4e/61a6f4e78fe6700f66b699ab38377f0aa5b33e3ef55751ba38
Successfully built scorecardpy


**import library**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

import scorecardpy as sc


**Load Dataset**

In [ ]:
df = pd.read_csv("bnk_credit.csv")

print(df.shape)
df.head()


(23896, 93)


,Unnamed: 0,dt_opened,customer_no,entry_time,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,...,bad_label,cur_balance_amt,creditlimit,cashlimit,amt_past_due,actualpaymentamount,ratio_currbalance_creditlimit,payment_duration,enq_amt,days_since_enquiry
0,0,101,0,117,2,109,41,2,1,1,...,0,4714857,670000.0,168000.0,2538209.0,35543.0,7.037089,1496.000000,2.767306e+05,5388.500000
1,1,120,11111,199,2,187,151,1,1,1,...,0,30754,1000000.0,1.0,0.0,0.0,0.030754,2035.000000,9.965122e+07,4720.940299
2,2,128,16119,199,2,187,165,1,1,1,...,0,17864,0.0,0.0,0.0,0.0,17864.000000,2454.000000,3.400000e+06,4379.000000
3,3,150,17230,139,2,178,161,1,1,1,...,0,1845569,956000.0,361000.0,0.0,1152.0,1.930509,2757.142857,1.509394e+06,5841.303030
4,4,33,18341,283,2,0,0,3,1,1,...,0,7973,0.0,0.0,0.0,0.0,7973.000000,678.500000,1.000000e+03,5424.500000


In [ ]:
df['bad_label'].value_counts()

,count
bad_label,
0,22892
1,1004


**TARGET**

In [ ]:
# Drop identifiers
df = df.drop(['dt_opened', 'customer_no', 'entry_time'], axis=1)

# Target
y = df['bad_label']
X = df.drop('bad_label', axis=1)


**TRAIN TEST SPLIT**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_df = X_train.copy()
train_df['bad_label'] = y_train

test_df = X_test.copy()
test_df['bad_label'] = y_test


**WOE BINNING**

In [ ]:
bins = sc.woebin(train_df, y="bad_label")

# Apply WOE transform
train_woe = sc.woebin_ply(train_df, bins)
test_woe = sc.woebin_ply(test_df, bins)

y_train = train_woe['bad_label']
X_train = train_woe.drop('bad_label', axis=1)

y_test = test_woe['bad_label']
X_test = test_woe.drop('bad_label', axis=1)


[INFO] creating woe binning ...


/usr/local/lib/python3.12/dist-packages/scorecardpy/condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
/usr/local/lib/python3.12/dist-packages/scorecardpy/condition_fun.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  datetime_cols = dat.apply(pd.to_numeric,errors='ignore').select_dtypes(object).apply(pd.to_datetime,errors='ignore').select_dtypes('datetime64').columns.tolist()
/usr/local/lib/python3.12/dist-packages/scorecardpy/woebin.py:320: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=Fals

Binning on 19116 rows and 90 columns in 00:00:53
[INFO] converting into woe values ...
[INFO] converting into woe values ...


**IV FEATURE SELECTION**

In [ ]:
# Calculate IV on original train
iv = sc.iv(train_df, y="bad_label")

# keep useful variables
selected_features = iv[iv['info_value'] > 0.02]['variable'].tolist()

# IMPORTANT: convert to WOE column names
selected_woe_features = [col + "_woe" for col in selected_features]

# filter only existing columns (safe)
selected_woe_features = [col for col in selected_woe_features if col in X_train.columns]

print("Selected WOE Features:", len(selected_woe_features))

# apply selection
X_train = X_train[selected_woe_features]
X_test = X_test[selected_woe_features]


Selected WOE Features: 59


**IMBALANCE FIX (SMOTE)**

In [ ]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", np.bincount(y_train))

smote = SMOTE(sampling_strategy=0.4, random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("After SMOTE:", np.bincount(y_train_sm))


Before SMOTE: [18313   803]
After SMOTE: [18313  7325]


**LOGISTIC REGRESSION**

In [ ]:
model = LogisticRegression(max_iter=5000, class_weight='balanced')
model.fit(X_train, y_train)


LogisticRegression(class_weight='balanced', max_iter=5000)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, y_prob)
gini = 2*auc - 1

print("AUC:", round(auc,4))
print("GINI:", round(gini,4))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


AUC: 0.6812
GINI: 0.3625

Confusion Matrix:
 [[2933 1646]
 [  80  121]]

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.64      0.77      4579
           1       0.07      0.60      0.12       201

    accuracy                           0.64      4780
   macro avg       0.52      0.62      0.45      4780
weighted avg       0.94      0.64      0.75      4780



**HYPERPARAMETER TUNING**

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 5, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': ['balanced']
}

log = LogisticRegression(max_iter=8000)

grid = GridSearchCV(
    log,
    param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV AUC:", grid.best_score_)


Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Params: {'C': 0.1, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'liblinear'}
Best CV AUC: 0.6943052774743406


In [ ]:
best_log = grid.best_estimator_
best_log.fit(X_train, y_train)


LogisticRegression(C=0.1, class_weight='balanced', max_iter=8000,
                   solver='liblinear')

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import matplotlib.pyplot as plt

y_pred = best_log.predict(X_test)
y_prob = best_log.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, y_prob)
gini = 2*auc - 1

print("Tuned Logistic AUC:", round(auc,4))
print("Tuned Logistic GINI:", round(gini,4))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Tuned Logistic AUC: 0.683
Tuned Logistic GINI: 0.366

Confusion Matrix:
 [[2927 1652]
 [  80  121]]

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.64      0.77      4579
           1       0.07      0.60      0.12       201

    accuracy                           0.64      4780
   macro avg       0.52      0.62      0.45      4780
weighted avg       0.94      0.64      0.74      4780



**XGBOOST MODEL**

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import matplotlib.pyplot as plt


In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=30,
    reg_alpha=0.5,
    reg_lambda=1.0,
    eval_metric='auc',
    random_state=42
)

xgb_model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=30, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=400,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, y_prob)
gini = 2*auc - 1

print("XGBoost AUC:", round(auc,4))
print("XGBoost GINI:", round(gini,4))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


XGBoost AUC: 0.675
XGBoost GINI: 0.3501

Confusion Matrix:
 [[4579    0]
 [ 201    0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      4579
           1       0.00      0.00      0.00       201

    accuracy                           0.96      4780
   macro avg       0.48      0.50      0.49      4780
weighted avg       0.92      0.96      0.94      4780



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**RANDOM SEARCH TUNING(XGBOOST MODEL)**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'max_depth': [2,3,4,5],
    'learning_rate': [0.01,0.03,0.05,0.08],
    'n_estimators': [300,500,700],
    'subsample': [0.7,0.8,0.9],
    'colsample_bytree': [0.6,0.7,0.8,0.9],
    'min_child_weight': [10,20,30,50],
    'gamma': [0,0.1,0.2],
    'reg_alpha': [0,0.5,1],
    'reg_lambda': [1,2,5]
}

xgb_base = xgb.XGBClassifier(eval_metric='auc', random_state=42)

search = RandomizedSearchCV(
    xgb_base,
    param_grid,
    scoring='roc_auc',
    n_iter=25,
    cv=4,
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best Params:", search.best_params_)
print("Best CV AUC:", search.best_score_)

best_xgb = search.best_estimator_


Fitting 4 folds for each of 25 candidates, totalling 100 fits
Best Params: {'subsample': 0.7, 'reg_lambda': 2, 'reg_alpha': 0.5, 'n_estimators': 700, 'min_child_weight': 20, 'max_depth': 4, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.6}
Best CV AUC: 0.6806380303401196


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Predict
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

# Metrics
auc = roc_auc_score(y_test, y_prob)
gini = 2*auc - 1

print("Tuned XGBoost AUC :", round(auc,4))
print("Tuned XGBoost GINI:", round(gini,4))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Tuned XGBoost AUC : 0.6826
Tuned XGBoost GINI: 0.3652

Confusion Matrix:
 [[4579    0]
 [ 201    0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      4579
           1       0.00      0.00      0.00       201

    accuracy                           0.96      4780
   macro avg       0.48      0.50      0.49      4780
weighted avg       0.92      0.96      0.94      4780



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Stacking (Hybrid Model)**
**(tuned logistic & tuned xgboost)**




In [ ]:
# Logistic probability
log_train_prob = best_log.predict_proba(X_train)[:,1]
log_test_prob  = best_log.predict_proba(X_test)[:,1]

# XGB probability
xgb_train_prob = best_xgb.predict_proba(X_train)[:,1]
xgb_test_prob  = best_xgb.predict_proba(X_test)[:,1]


In [ ]:
import pandas as pd

stack_train = pd.DataFrame({
    'logistic': log_train_prob,
    'xgboost': xgb_train_prob
})

stack_test = pd.DataFrame({
    'logistic': log_test_prob,
    'xgboost': xgb_test_prob
})


In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression()
meta_model.fit(stack_train, y_train)


LogisticRegression()

In [ ]:
final_prob = meta_model.predict_proba(stack_test)[:,1]
final_pred = (final_prob > 0.5).astype(int)


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import matplotlib.pyplot as plt

auc = roc_auc_score(y_test, final_prob)
gini = 2*auc - 1

print("STACKED MODEL AUC:", round(auc,4))
print("STACKED MODEL GINI:", round(gini,4))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, final_pred))
print("\nClassification Report:\n", classification_report(y_test, final_pred))


STACKED MODEL AUC: 0.6842
STACKED MODEL GINI: 0.3684

Confusion Matrix:
 [[4579    0]
 [ 201    0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      4579
           1       0.00      0.00      0.00       201

    accuracy                           0.96      4780
   macro avg       0.48      0.50      0.49      4780
weighted avg       0.92      0.96      0.94      4780



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**RANK ORDERING**

In [ ]:
scorecard_df = pd.DataFrame({
    "actual": y_test,
    "probability": final_prob
})

scorecard_df = scorecard_df.sort_values(by='probability', ascending=False)
scorecard_df['decile'] = pd.qcut(scorecard_df['probability'], 10, labels=False) + 1


In [ ]:
rank_table = scorecard_df.groupby('decile').agg(
    total_customers=('actual','count'),
    bad_customers=('actual','sum')
).reset_index()

rank_table['good_customers'] = rank_table['total_customers'] - rank_table['bad_customers']
rank_table['bad_rate'] = rank_table['bad_customers'] / rank_table['total_customers']

rank_table = rank_table.sort_values(by='decile', ascending=False)
rank_table


,decile,total_customers,bad_customers,good_customers,bad_rate
9,10,478,45,433,0.094142
8,9,478,33,445,0.069038
7,8,478,30,448,0.062762
6,7,478,21,457,0.043933
5,6,478,26,452,0.054393
4,5,478,18,460,0.037657
3,4,478,10,468,0.020921
2,3,478,6,472,0.012552
1,2,478,7,471,0.014644
0,1,478,5,473,0.010460


**Save All Models**

In [ ]:
import joblib

# save models
joblib.dump(best_log, "logistic_model.pkl")
joblib.dump(best_xgb, "xgb_model.pkl")
joblib.dump(meta_model, "stack_model.pkl")
joblib.dump(bins, "woe_bins.pkl")
joblib.dump(selected_woe_features, "features.pkl")


['features.pkl']